# Tile Card Studio (Colab)
صفحة ويب كاملة للمستخدم: يكتب اسم شخصية ← Gemini يصمّم ← يعدّل بتعليق ← يغيّر اللون والاسم ← يحمّل. وكل الكروت بتتحفظ في مكتبة.

**قبل ما تبدئي:** (1) API key من https://aistudio.google.com/apikey  (2) ملف `tilesys_pkg.zip` الجديد (فيه فولدر `app`).
شغّلي الخلايا بالترتيب.

In [ ]:
# 1) تثبيت المكتبات (مفيش playwright خلاص، التحميل بيحصل في المتصفح)
!pip -q install flask cairosvg google-genai

In [ ]:
# 2) ارفعي tilesys_pkg.zip
from google.colab import files
import os, zipfile
up = files.upload()
os.makedirs('/content/tilesys', exist_ok=True)
zipfile.ZipFile(list(up.keys())[0]).extractall('/content/tilesys')
os.chdir('/content/tilesys')
print(os.listdir('.'))

In [ ]:
# 3) مفتاح Gemini (بيتكتب مخفي)
from getpass import getpass
API_KEY = getpass('Gemini API key: ')

In [ ]:
# 4) (اختياري، يُنصح به) احفظي المكتبة على Google Drive عشان الكروت ماتضيعش لما Colab يقفل
USE_DRIVE = True
LIB_DIR = '/content/tilesys/library'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    LIB_DIR = '/content/drive/MyDrive/tile_card_library'
print('المكتبة في:', LIB_DIR)

In [ ]:
# 5) شغّلي السيرفر
import sys, os
os.chdir('/content/tilesys'); sys.path.insert(0, '/content/tilesys/app'); sys.path.insert(0, '/content/tilesys')
import importlib, server
importlib.reload(server)
PORT = 8000
if 'app' in globals() and getattr(app, 'config', {}).get('_srv'):          # لو شغّلتيها قبل كده
    app.config['_srv'].shutdown()
app = server.create_app(api_key=API_KEY, lib_dir=LIB_DIR,
                        workers=2,          # كام تصميم بالتوازي (زوّديه لو الباقة المدفوعة)
                        rate_limit=60,      # أقصى عدد طلبات (شخصية/تعديل) لكل مستخدم في الساعة (0 = من غير حد)
                        parallel_jobs=1)    # كام شخصية تتعمل مع بعض. 1 للمجاني، و2 أو 3 للباقة المدفوعة
server.start_server(app, port=PORT)
print('السيرفر اشتغل')

In [ ]:
# 6) افتحي الصفحة (في تبويب جديد أحسن عشان التحميل يشتغل صح)
from google.colab import output
output.serve_kernel_port_as_window(8000)
# لو عايزاها جوه النوت بوك بدل تبويب:  output.serve_kernel_port_as_iframe(8000, height=1000)

### (اختياري) رابط عام تبعتيه لمستخدمين تانيين
الخلية دي بتعمل رابط `trycloudflare.com` مؤقت للصفحة. **ما تنشريش الرابط على نطاق واسع** لأن كل توليد بيستهلك من حصة مفتاحك. وهو شغال طول ما النوت بوك شغال بس.

In [ ]:
import subprocess, re, time
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
log = open('cf.log', 'w')
subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=log, stderr=log)
for _ in range(30):
    time.sleep(1)
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', open('cf.log').read())
    if m: print('الرابط العام:', m.group(0)); break
else:
    print('مظهرش رابط، شوفي cf.log')